### **Feature Engineering: Construction and Splitting**

Topic Roadmap

**1. Imports**

**2. Feature Construction**
- **2.1 Dataset Loading & Baseline Evaluation**
- **2.2 Constructing New Features (`Family_size`, `Family_type`)**
- **2.3 Evaluating Constructed Features**

**3. Feature Splitting**
- **3.1 Extracting Titles from Text Data**
- **3.2 Deriving Binary Features from Splits**

**4. Key Revision Notes**

### **1. Imports**

Import standard libraries and necessary tools from scikit-learn

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

### **2. Feature Construction**

Feature construction involves combining or transforming existing features into new, more meaningful representations

**2.1 Dataset Loading & Baseline Evaluation**

First, we load the relevant columns, drop missing values, and establish a baseline accuracy score using 20-fold cross-validation

In [ ]:
# Load data and drop missing values
cols = ['Age', 'Pclass', 'SibSp', 'Parch', 'Survived']
df_const = pd.read_csv('docs/Lecture-031-train.csv')[cols]
df_const.dropna(inplace=True)

# Isolate features (X) and target (y)
X_const = df_const.iloc[:, 0:4]
y_const = df_const.iloc[:, -1]

In [3]:
# Establish a baseline accuracy score using Logistic Regression
baseline_score = np.mean(cross_val_score(LogisticRegression(), X_const, y_const, scoring='accuracy', cv=20))
print(f"Baseline Accuracy: {baseline_score:.4f}")

Baseline Accuracy: 0.6933


**2.2 Constructing New Features (`Family_size`, `Family_type`)**

We can combine the `SibSp` (siblings/spouses) and `Parch` (parents/children) columns to calculate the total `Family_size`[cite: 11]. We then categorize this size into a `Family_type`

In [5]:
# Construct Family_size (+1 includes the passenger themselves)
X_const['Family_size'] = X_const['SibSp'] + X_const['Parch'] + 1
X_const.head()

,Age,Pclass,SibSp,Parch,Family_size
0,22.0,3,1,0,2
1,38.0,1,1,0,2
2,26.0,3,0,0,1
3,35.0,1,1,0,2
4,35.0,3,0,0,1


In [6]:
# Define a function to categorize family size
def categorize_family(num):
    if num == 1:
        return 0  # Alone
    elif num > 1 and num <= 4:
        return 1  # Small family
    else:
        return 2  # Large family

# Apply the function to construct Family_type
X_const['Family_type'] = X_const['Family_size'].apply(categorize_family)
X_const.head()

,Age,Pclass,SibSp,Parch,Family_size,Family_type
0,22.0,3,1,0,2,1
1,38.0,1,1,0,2,1
2,26.0,3,0,0,1,0
3,35.0,1,1,0,2,1
4,35.0,3,0,0,1,0


**2.3 Evaluating Constructed Features**

Drop the redundant original features and re-evaluate the model to see if the new features improved performance

In [7]:
# Drop original columns that are now represented by Family_type
X_const.drop(columns=['SibSp', 'Parch', 'Family_size'], inplace=True)

# Re-evaluate model accuracy
new_score = np.mean(cross_val_score(LogisticRegression(), X_const, y_const, scoring='accuracy', cv=20))
print(f"Accuracy with Constructed Features: {new_score:.4f}")

Accuracy with Constructed Features: 0.7003


### **3. Feature Splitting**

Feature splitting involves extracting multiple discrete pieces of information from a single string or complex variable[cite: 11].

**3.1 Extracting Titles from Text Data**

We can split the `Name` column to extract the passenger's title (e.g., Mr., Mrs., Miss)[cite: 11].

In [13]:
# Load full dataset for string manipulation
df_split = pd.read_csv('docs/Lecture-031-train.csv')

# Example of a name: "Braund, Mr. Owen Harris"
# 1. Split by ", " and take the second part: "Mr. Owen Harris"
# 2. Split by "." and take the first part: "Mr"
df_split['Title'] = df_split['Name'].str.split(', ', expand=True)[1].str.split('.', expand=True)[0]

df_split[['Name', 'Title']].head()

,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr


**3.2 Deriving Binary Features from Splits**

After extracting the title, we can analyze its relationship with the target variable and construct new binary indicators

In [14]:
# Check survival rates by Title
title_survival = df_split.groupby('Title')['Survived'].mean().sort_values(ascending=False)
print("Survival Rate by Title:\n", title_survival.head())

Survival Rate by Title:
 Title
Lady            1.0
Ms              1.0
Sir             1.0
Mme             1.0
the Countess    1.0
Name: Survived, dtype: float64


In [15]:
# Construct an 'Is_Married' feature based on the 'Mrs' title
df_split['Is_Married'] = 0
df_split.loc[df_split['Title'] == 'Mrs', 'Is_Married'] = 1

print("Value counts for Is_Married:\n", df_split['Is_Married'].value_counts())

Value counts for Is_Married:
 Is_Married
0    766
1    125
Name: count, dtype: int64


### **4. Key Revision Notes**

- **Feature Construction:** Mathematical or logical combination of existing features (e.g., adding distinct family member counts to get total family size). It helps models capture non-linear relationships or interactions easily.
- **Feature Splitting:** Using string operations (like `str.split(expand=True)`) to isolate buried categories, such as extracting titles or surnames from a full name string.
- **Dimensionality Management:** Always remember to drop the original features after successfully capturing their information in constructed features to prevent multicollinearity and reduce noise.